In [1]:
import os
import pandas as pd
import numpy as np
import duckdb

In [7]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand AS

SELECT
    DATE_TRUNC(
        'hour',
        tpep_dropoff_datetime
    ) AS dropoff_hour,
    
    DOLocationID,
    COUNT(*) AS demand

FROM 'data/taxi.parquet'

WHERE
    tpep_dropoff_datetime IS NOT NULL
    AND DOLocationID IS NOT NULL
    
    AND YEAR(tpep_dropoff_datetime) = 2023

GROUP BY
    dropoff_hour,
    DOLocationID

ORDER BY
    dropoff_hour,
    DOLocationID
""")

In [3]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand_features AS

SELECT
    dropoff_hour,
    DOLocationID,
    demand,

    MONTH(dropoff_hour) AS month,
    DAY(dropoff_hour) AS day,
    HOUR(dropoff_hour) AS hour,
    DAYOFWEEK(dropoff_hour) AS weekday,

    CASE
        WHEN DAYOFWEEK(dropoff_hour) IN (0, 6)
        THEN 1
        ELSE 0
    END AS is_weekend

FROM taxi_demand

ORDER BY
    dropoff_hour,
    DOLocationID
""")

In [4]:
duckdb.sql("""
COPY taxi_demand_features
TO 'data/taxi_demand_processed_do.parquet'
(FORMAT PARQUET)
""")

In [5]:
print(
    duckdb.sql("""
    SELECT COUNT(*) AS total_rows
    FROM 'data/taxi_demand_processed_do.parquet'
    """).fetchall()
)

print(
    duckdb.sql("""
    SELECT *
    FROM 'data/taxi_demand_processed_do.parquet'
    LIMIT 10
    """).df()
)

[(1397293,)]
  dropoff_hour  DOLocationID  demand  month  day  hour  weekday  is_weekend
0   2023-01-01             3       1      1    1     0        0           1
1   2023-01-01             4      15      1    1     0        0           1
2   2023-01-01             7      19      1    1     0        0           1
3   2023-01-01             9       2      1    1     0        0           1
4   2023-01-01            10       4      1    1     0        0           1
5   2023-01-01            13      24      1    1     0        0           1
6   2023-01-01            14       2      1    1     0        0           1
7   2023-01-01            15       1      1    1     0        0           1
8   2023-01-01            17       3      1    1     0        0           1
9   2023-01-01            18       2      1    1     0        0           1


In [6]:
print(
    duckdb.sql("""
    SELECT
        COUNT(*) AS n,
        MIN(demand) AS min_demand,
        MAX(demand) AS max_demand,
        AVG(demand) AS mean_demand,
        STDDEV(demand) AS std_demand,
        MEDIAN(demand) AS median_demand
    FROM 'data/taxi_demand_processed_do.parquet'
    """).df()
)

         n  min_demand  max_demand  mean_demand  std_demand  median_demand
0  1397293           1         656    27.417021   53.362016            4.0
